# 🏗️ Crane Predictive Maintenance & Root Cause Analysis - Enhanced Pipeline

This notebook demonstrates two complementary analytical stories on crane sensor data, framed after the **Liebherr LiDAT** telematics system:

| Story | Focus | Question answered | LiDAT feature |
|-------|--------|-------------------|---------------|
| **Root Cause Analysis (RCA)** | Reactive | *Why did it fail?* | Safety Reports · Sensor Notifications · Teleservice |
| **Predictive Maintenance (PdM)** | Proactive | *When will it fail?* | Operating Hours (Bh) · Usage Trends · Availability |

> **Key insight:** Both stories share the same dataset. RCA reads sensor snapshots at the moment of a fault; PdM reads the same sensors as a time series to project forward.

**Four enhancements over the baseline notebook:**

| # | Module | Purpose |
|---|--------|---------|
| 1 | **Rolling feature engineering** | Capture temporal degradation signals (LiDAT: Usage Trends) |
| 2 | **Hyperparameter tuning (Optuna)** | Bayesian search for optimal XGBoost parameters |
| 3 | **Confusion matrix & evaluation visuals** | Deep-dive classification diagnostics |
| 4 | **Model serialization (joblib)** | Save and reload the full inference pipeline |

**Prerequisites:**
```bash
pip install pandas numpy scikit-learn xgboost optuna matplotlib seaborn joblib
```

## 0. Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import optuna
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.preprocessing import label_binarize
from xgboost import XGBClassifier
from sklearn.linear_model import LinearRegression

optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("✅ Imports complete.")

## 1. Synthetic Dataset Generation

We generate a realistic crane sensor dataset inline so the notebook is fully self-contained.

**Column mapping to LiDAT concepts:**

| Column | LiDAT concept | Used by |
|--------|---------------|---------|
| `Timestamp` | Operating Hours (Bh) | PdM — time axis for wear regression |
| `Load_kg` | Safety Reports | RCA — overload detection |
| `Motor_Temp` | Sensor Notifications | RCA — overheat alerts |
| `Vibration` | Sensor Notifications | RCA — mechanical stress signals |
| `Brake_Wear` | Usage Trends | PdM — degradation trajectory |
| `Error_Code` | Teleservice fault log | RCA — classification target |

In [ ]:
def generate_crane_data(n: int = 2000, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    timestamps = pd.date_range("2024-01-01", periods=n, freq="h")

    load   = rng.uniform(500, 5000, n)
    temp   = 60 + load / 200 + rng.normal(0, 5, n)
    vib    = 2 + load / 3000 + rng.normal(0, 0.4, n)
    # Brake wear: monotonically decreasing, load-dependent rate
    wear_rate  = 0.001 + load / 5_000_000
    brake_wear = np.maximum(10 - np.cumsum(wear_rate) + rng.normal(0, 0.05, n), 0.5)

    def label(row):
        if row["Load_kg"]    > 4500: return "E01_Overload"
        if row["Motor_Temp"] > 95:   return "E02_Overheat"
        if row["Vibration"]  > 3.8:  return "E03_HighVibration"
        if row["Brake_Wear"] < 2.0:  return "E04_BrakeWear"
        return "OK"

    df = pd.DataFrame({
        "Timestamp":  timestamps,
        "Load_kg":    load.round(1),
        "Motor_Temp": temp.round(2),
        "Vibration":  vib.round(3),
        "Brake_Wear": brake_wear.round(3),
    })
    df["Error_Code"] = df.apply(label, axis=1)
    return df


df_raw = generate_crane_data(n=2000)
print(f"Dataset shape : {df_raw.shape}")
print(f"Date range    : {df_raw.Timestamp.min()} → {df_raw.Timestamp.max()}")
print(f"\nFault distribution (LiDAT Error_Code / Teleservice log):")
print(df_raw.Error_Code.value_counts())
df_raw.head()

---
## STORY A - Root Cause Analysis (RCA)
### *Reactive: Why did the crane fault?*

**LiDAT equivalent:** Safety Reports + Sensor Notifications + Teleservice remote diagnosis.

We train an XGBoost classifier to identify the `Error_Code` from a sensor snapshot at the time of a fault event. This mirrors what a Liebherr service technician would do via Teleservice: look at the sensor state at the moment of the alert and determine the root cause.

**Pipeline:**
1. Rolling feature engineering — captures instability in the hours *before* a fault
2. Chronological train/test split
3. Optuna hyperparameter tuning
4. Classification diagnostics (confusion matrix, per-class F1, ROC curves)

### A1. Rolling Feature Engineering

**LiDAT concept: Usage Trends** — monitoring sensor behaviour over time, not just at a point in time.

Raw sensor readings miss patterns that precede faults. We add 12-h and 24-h sliding-window statistics (mean, std, min, max) for each sensor. The rolling standard deviation is particularly informative for RCA: a spike in `Vibration_roll12_std` signals mechanical instability that may not yet have triggered a fault code.

In [ ]:
SENSORS = ["Load_kg", "Motor_Temp", "Vibration", "Brake_Wear"]
WINDOWS = [12, 24]

def add_rolling_features(df: pd.DataFrame, windows: list, sensors: list) -> pd.DataFrame:
    df = df.copy().sort_values("Timestamp").reset_index(drop=True)
    for sensor in sensors:
        for w in windows:
            rolled = df[sensor].rolling(window=w, min_periods=1)
            base = f"{sensor}_roll{w}"
            df[f"{base}_mean"] = rolled.mean()
            df[f"{base}_std"]  = rolled.std().fillna(0)
            df[f"{base}_min"]  = rolled.min()
            df[f"{base}_max"]  = rolled.max()
    return df

df = add_rolling_features(df_raw, windows=WINDOWS, sensors=SENSORS)
rolling_cols = [c for c in df.columns if "_roll" in c]

print(f"Raw features    : {len(SENSORS)}")
print(f"Rolling features: {len(rolling_cols)}")
print(f"Total features  : {len(SENSORS) + len(rolling_cols)}")

In [ ]:
# Visualise: raw Vibration vs its rolling means
# LiDAT analogy: sensor notification trend line in the LiDAT dashboard
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df["Timestamp"], df["Vibration"], alpha=0.3, lw=0.8, label="Raw Vibration")
ax.plot(df["Timestamp"], df["Vibration_roll12_mean"], lw=1.5, label="12-h rolling mean")
ax.plot(df["Timestamp"], df["Vibration_roll24_mean"], lw=1.5, label="24-h rolling mean")
ax.set_title("[RCA / LiDAT Sensor Notifications] Vibration — Raw vs Rolling Means")
ax.set_xlabel("Timestamp (Operating Hours proxy)")
ax.set_ylabel("Vibration (mm/s)")
ax.legend()
plt.tight_layout()
plt.show()

### A2. Train / Test Split & Preprocessing

We use a **chronological 80/20 split** — no shuffling — to respect the time-series nature of the data. This reflects real-world deployment: a model trained on historical faults must generalise to future faults.

In [ ]:
FEATURE_COLS = SENSORS + rolling_cols
TARGET_COL   = "Error_Code"

split_idx   = int(len(df) * 0.8)
train_df    = df.iloc[:split_idx]
test_df     = df.iloc[split_idx:]

le      = LabelEncoder()
y_train = le.fit_transform(train_df[TARGET_COL])
y_test  = le.transform(test_df[TARGET_COL])

scaler  = StandardScaler()
X_train = scaler.fit_transform(train_df[FEATURE_COLS])
X_test  = scaler.transform(test_df[FEATURE_COLS])

print(f"Train : {len(X_train)} samples | Test : {len(X_test)} samples")
print(f"Classes: {list(le.classes_)}")

### A3. Hyperparameter Tuning with Optuna

Optuna uses **Tree-structured Parzen Estimator (TPE)** — a Bayesian optimisation strategy. Each trial trains an XGBoost model and evaluates it via 3-fold stratified cross-validation.

| Parameter | Search range | Effect |
|-----------|-------------|--------|
| `n_estimators` | 100–600 | Model capacity |
| `max_depth` | 3–10 | Tree complexity |
| `learning_rate` | 0.01–0.3 | Shrinkage per step |
| `subsample` | 0.5–1.0 | Row sampling |
| `colsample_bytree` | 0.5–1.0 | Feature sampling per tree |
| `min_child_weight` | 1–10 | Minimum leaf node weight |

In [ ]:
def objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 100, 600, step=50),
        "max_depth":        trial.suggest_int("max_depth", 3, 10),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "eval_metric":      "mlogloss",
        "random_state":     RANDOM_STATE,
    }
    clf    = XGBClassifier(**params)
    cv     = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(clf, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1)
    return scores.mean()


study = optuna.create_study(direction="maximize", study_name="crane_rca_tuning")
study.optimize(objective, n_trials=40, show_progress_bar=True)

print(f"\n✅ Best trial    : #{study.best_trial.number}")
print(f"   Best F1-macro : {study.best_value:.4f}")
print(f"   Best params   : {study.best_params}")

In [ ]:
# Optimisation history
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

trial_values = [t.value for t in study.trials]
running_best = pd.Series(trial_values).cummax()
axes[0].plot(trial_values, alpha=0.5, label="Trial F1")
axes[0].plot(running_best, lw=2, label="Running best")
axes[0].set_title("[RCA] Optuna — Optimisation history")
axes[0].set_xlabel("Trial")
axes[0].set_ylabel("F1-macro (3-fold CV)")
axes[0].legend()

param_names  = list(study.best_params.keys())
param_values = [str(v) for v in study.best_params.values()]
axes[1].barh(param_names, [float(v) if v.replace('.','',1).isdigit() else 1 for v in param_values],
             color="steelblue")
axes[1].set_title("Best hyperparameters")

plt.tight_layout()
plt.show()

### A4. Train Final RCA Model & Evaluate

**LiDAT analogy:** This is the Teleservice diagnostic — given a sensor snapshot, predict the fault code remotely.

In [ ]:
best_params = {**study.best_params, "eval_metric": "mlogloss", "random_state": RANDOM_STATE}

clf_rca = XGBClassifier(**best_params)
clf_rca.fit(X_train, y_train)

y_pred      = clf_rca.predict(X_test)
y_pred_prob = clf_rca.predict_proba(X_test)

present_labels = sorted(set(y_train) | set(y_test))
present_names  = [le.classes_[i] for i in present_labels]

print("=== [RCA] Classification Report — Tuned XGBoost ===")
print(classification_report(y_test, y_pred, labels=present_labels,
                            target_names=present_names, zero_division=0))

### A5. Confusion Matrix & Evaluation Visuals

Three diagnostic plots:
1. **Normalised confusion matrix** — which faults are being confused with each other?
2. **Per-class F1 bar chart** — which fault codes are hardest to detect?
3. **One-vs-Rest ROC curves** — discrimination power per fault class

In [ ]:
# Confusion matrices
cm      = confusion_matrix(y_test, y_pred, labels=present_labels)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay(cm, display_labels=present_names).plot(
    ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("[RCA] Confusion matrix — raw counts")
axes[0].tick_params(axis="x", rotation=30)

ConfusionMatrixDisplay(cm_norm.round(2), display_labels=present_names).plot(
    ax=axes[1], colorbar=False, cmap="Blues")
axes[1].set_title("[RCA] Confusion matrix — row-normalised")
axes[1].tick_params(axis="x", rotation=30)

plt.suptitle("Root Cause Analysis — Fault Classification Diagnostics", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "rca_confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Per-class F1 bar chart
f1_per_class = f1_score(y_test, y_pred, labels=present_labels, average=None, zero_division=0)
f1_df = pd.DataFrame({"Fault": present_names, "F1": f1_per_class}).sort_values("F1")

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(f1_df["Fault"], f1_df["F1"],
               color=["#d9534f" if v < 0.80 else "#5cb85c" for v in f1_df["F1"]])
ax.axvline(0.80, color="grey", linestyle="--", lw=1, label="0.80 threshold")
ax.bar_label(bars, fmt="%.3f", padding=3)
ax.set_xlim(0, 1.05)
ax.set_title("[RCA] Per-class F1 — Tuned XGBoost")
ax.set_xlabel("F1 score")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "rca_f1_per_class.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# One-vs-Rest ROC curves
y_test_bin = label_binarize(y_test, classes=present_labels)

fig, ax = plt.subplots(figsize=(8, 6))
for i, (label_name, color) in enumerate(zip(present_names, plt.cm.tab10.colors)):
    if y_test_bin[:, i].sum() == 0:
        continue
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_prob[:, i])
    auc = roc_auc_score(y_test_bin[:, i], y_pred_prob[:, i])
    ax.plot(fpr, tpr, lw=2, color=color, label=f"{label_name} (AUC={auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random")
ax.set_title("[RCA] One-vs-Rest ROC curves — Fault classification")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "rca_roc_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Feature importance — which sensors drive RCA predictions?
importances = clf_rca.feature_importances_
feat_df = (
    pd.DataFrame({"Feature": FEATURE_COLS, "Importance": importances})
    .sort_values("Importance", ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=feat_df, x="Importance", y="Feature",
            hue="Feature", palette="viridis", legend=False, ax=ax)
ax.set_title("[RCA] Top-20 Feature Importances — Tuned XGBoost")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "rca_feature_importance.png", dpi=120, bbox_inches="tight")
plt.show()

---
## STORY B - Predictive Maintenance (PdM)
### *Proactive: When will the brake pad fail?*

**LiDAT equivalent:** Operating Hours (Bh) tracking + Usage Trends + Availability planning.

Instead of classifying a fault that has already happened, we project the `Brake_Wear` time series forward to predict the date it crosses the critical threshold of **1.0 mm** — the point at which a maintenance intervention must be scheduled to preserve availability.

This mirrors LiDAT's availability goal: prepare spare parts and dispatch a service technician *just-in-time*, before a forced downtime event.

We use **linear regression** as a transparent, auditable baseline — appropriate for a monotonically decreasing wear curve with roughly constant load.

In [ ]:
# Use row index as operating-hours proxy (LiDAT: Bh meter)
hours      = np.arange(len(df)).reshape(-1, 1)
brake_wear = df["Brake_Wear"].values

reg = LinearRegression()
reg.fit(hours, brake_wear)

print(f"Slope (mm per operating hour) : {reg.coef_[0]:.6f}")
print(f"Intercept                      : {reg.intercept_:.4f}")

# Predict: when does wear reach 1.0 mm?
# 1.0 = slope * t + intercept  →  t = (1.0 - intercept) / slope
CRITICAL_MM    = 1.0
t_critical     = (CRITICAL_MM - reg.intercept_) / reg.coef_[0]
t_critical_dt  = df["Timestamp"].iloc[0] + pd.Timedelta(hours=t_critical)
hours_remaining = int(t_critical - len(df))

print(f"\n⚠️  [PdM / LiDAT Availability] Predicted maintenance date:")
print(f"    {t_critical_dt.strftime('%Y-%m-%d %H:%M')}")
print(f"    (~{hours_remaining} operating hours from last observation)")

In [ ]:
# Visualise brake wear trend — LiDAT Usage Trends chart equivalent
future_hours = np.arange(int(t_critical) + 50).reshape(-1, 1)
predicted_wear = reg.predict(future_hours)
future_ts = [
    df["Timestamp"].iloc[0] + pd.Timedelta(hours=int(h)) for h in future_hours.flatten()
]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df["Timestamp"], brake_wear, alpha=0.7, label="Measured Brake_Wear")
ax.plot(future_ts, predicted_wear, "--", color="orange", label="Linear trend (PdM projection)")
ax.axhline(CRITICAL_MM, color="red", linestyle=":", label=f"Critical threshold ({CRITICAL_MM} mm)")
ax.axvline(t_critical_dt, color="red", linestyle="--", alpha=0.6,
           label=f"Maintenance due: {t_critical_dt.strftime('%Y-%m-%d')}")
ax.set_xlabel("Timestamp (Operating Hours proxy — LiDAT Bh)")
ax.set_ylabel("Brake Wear (mm)")
ax.set_title("[PdM / LiDAT Usage Trends] Brake Pad Wear Forecast")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pdm_brake_forecast.png", dpi=120, bbox_inches="tight")
plt.show()

---
## 3. Model Serialization

We bundle the **scaler + RCA classifier + label encoder** into a single `joblib` artefact. In a production LiDAT-connected system, this bundle would be loaded by the Teleservice backend to score incoming sensor telemetry in real time.

**Saved artefacts:**
- `outputs/crane_rca_pipeline.joblib` — full RCA inference bundle
- `outputs/crane_pdm_regression.joblib` — PdM linear regression model
- `outputs/feature_cols.txt` — ordered feature list (required for column alignment)

In [ ]:
# Save RCA pipeline
rca_bundle = {
    "scaler":         scaler,
    "classifier":     clf_rca,
    "label_encoder":  le,
    "feature_cols":   FEATURE_COLS,
    "best_params":    study.best_params,
    "cv_f1_macro":    study.best_value,
    "story":          "RCA — fault classification (LiDAT Teleservice)",
}
rca_path = OUTPUT_DIR / "crane_rca_pipeline.joblib"
joblib.dump(rca_bundle, rca_path, compress=3)
print(f"✅ RCA pipeline saved → {rca_path}  ({rca_path.stat().st_size / 1024:.1f} KB)")

# Save PdM regression
pdm_bundle = {
    "regressor":         reg,
    "critical_threshold": CRITICAL_MM,
    "predicted_date":    str(t_critical_dt),
    "story":             "PdM — brake wear forecast (LiDAT Operating Hours / Usage Trends)",
}
pdm_path = OUTPUT_DIR / "crane_pdm_regression.joblib"
joblib.dump(pdm_bundle, pdm_path, compress=3)
print(f"✅ PdM model saved  → {pdm_path}  ({pdm_path.stat().st_size / 1024:.1f} KB)")

# Feature list
feat_path = OUTPUT_DIR / "feature_cols.txt"
feat_path.write_text("\n".join(FEATURE_COLS))
print(f"✅ Feature list     → {feat_path}")

In [ ]:
# Reload RCA pipeline and run a sample inference
loaded_rca = joblib.load(rca_path)

sample = pd.DataFrame([{
    "Load_kg": 3500, "Motor_Temp": 88.0,
    "Vibration": 2.9, "Brake_Wear": 1.8,
}])
# Add rolling features — for a single point, rolling = same as raw; std = 0
for sensor in SENSORS:
    for w in WINDOWS:
        for stat in ["mean", "std", "min", "max"]:
            col = f"{sensor}_roll{w}_{stat}"
            sample[col] = sample[sensor] if stat != "std" else 0.0

X_sample    = loaded_rca["scaler"].transform(sample[loaded_rca["feature_cols"]])
pred_label  = loaded_rca["label_encoder"].inverse_transform(
    loaded_rca["classifier"].predict(X_sample)
)[0]
pred_proba  = loaded_rca["classifier"].predict_proba(X_sample)[0]

print("🔮 [RCA] Inference on reloaded pipeline")
print(f"   Predicted fault : {pred_label}")
print("   Class probabilities:")
for cls, prob in zip(loaded_rca["label_encoder"].classes_, pred_proba):
    print(f"     {cls:<22} {prob:.4f}")

---
## 4. Summary

| | RCA - Story A | PdM - Story B |
|---|---|---|
| **Question** | Why did it fail? | When will it fail? |
| **LiDAT feature** | Safety Reports · Sensor Notifications · Teleservice | Operating Hours (Bh) · Usage Trends · Availability |
| **ML method** | XGBoost classifier (tuned with Optuna) | Linear regression |
| **Key feature** | Rolling std of Vibration / Motor_Temp | Brake_Wear time series |
| **Output** | Predicted `Error_Code` + class probabilities | Predicted maintenance date |
| **Saved artefact** | `crane_rca_pipeline.joblib` | `crane_pdm_regression.joblib` |

### Next steps
- **SHAP** — add `shap.TreeExplainer` for per-prediction explanations on the RCA model
- **Drift detection** — monitor rolling feature distributions in production with `evidently`
- **MLflow** — replace `joblib` bundles with `mlflow.xgboost.log_model` for experiment versioning
- **Non-linear PdM** — replace linear regression with an exponential or polynomial decay model for more realistic brake wear curves
- **Load-weighted degradation** — weight brake wear rate by `Load_kg` per cycle, not just time